# Raw H5 Data → LeRobotDataset v2.1 Direct Conversion

이 노트북은 raw H5 파일을 직접 LeRobotDataset **v2.1 형식(episode-per-file)**으로 변환합니다.

v3를 거치지 않고 한 번에 변환하므로 더 효율적입니다.

핵심 목표:
- Raw H5 파일 읽기
- Image 데이터를 OpenPI-safe uint8 고정크기 텐서로 변환
- v2.1 폴더 레이아웃(`meta/`, `data/chunk-{}/`) 생성
- 메타데이터(info.json, episodes.jsonl, tasks.jsonl) 작성

## 1. 의존성 및 설정 로드

In [2]:
# 의존성 확인
import sys
import importlib
import yaml
import os

def _req(name: str):
    try:
        return importlib.import_module(name)
    except Exception as e:
        raise RuntimeError(f'Missing dependency: {name} ({e})')

np = _req('numpy')
h5py = _req('h5py')
pa = _req('pyarrow')
pq = _req('pyarrow.parquet')
PIL = _req('PIL')
Image = _req('PIL.Image')
tqdm = _req('tqdm.auto').tqdm

print('python:', sys.executable)
print('numpy:', np.__version__)
print('h5py:', h5py.__version__)
print('pyarrow:', pa.__version__)
print('Pillow:', PIL.__version__)

python: /home/hyunjin/miniforge3/envs/rby1/bin/python
numpy: 2.4.2
h5py: 3.15.1
pyarrow: 23.0.0
Pillow: 12.1.1


## 2. 경로 및 설정

In [3]:
from pathlib import Path
import json
import shutil
import logging

# config.yaml에서 설정 로드
with open(r'../config.yaml', encoding='utf-8') as f:
    config = yaml.safe_load(f)

root = config['demo_root']
task_name = config['conversion_task_name']
demo_root = os.path.join(root, task_name)
user_name = config['user_name']
v2_dataset_saving_root = config['v2_dataset_saving_root']
conversion_prompt = config.get('conversion_prompt', 'pick up the cup and place it on the plate')

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# 경로 설정
RAW_H5_DIR = Path(demo_root)
OUTPUT_V21_DATASET_DIR = Path(os.path.join(v2_dataset_saving_root, task_name))
os.makedirs(OUTPUT_V21_DATASET_DIR, exist_ok=True)

# v2.1 설정
REPO_ID = f"{user_name}/{task_name}"
DATASET_FPS = config['rec_fps']
IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
CHUNKS_SIZE = 1000  # v2.1 에피소드 청크 크기

print('RAW_H5_DIR:', RAW_H5_DIR)
print('OUTPUT_V21_DATASET_DIR:', OUTPUT_V21_DATASET_DIR)
print('DATASET_FPS:', DATASET_FPS)
print('conversion_prompt:', conversion_prompt)


RAW_H5_DIR: /media/hyunjin/T7/rby1_demo/PuttingCupintotheDishV2
OUTPUT_V21_DATASET_DIR: /media/hyunjin/T7/rby1_demo/LeRobotDataset_v2/PuttingCupintotheDishV2
DATASET_FPS: 15
conversion_prompt: pick up the cup and place it on the plate


## 3. Helper 함수

In [11]:
import io

def _write_json(path: Path, obj):
    """JSON 파일 작성"""
    path = Path(path)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False))

def _write_jsonl(path: Path, rows: list[dict]):
    """JSONL 파일 작성"""
    path = Path(path)
    with path.open('w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False))
            f.write('\n')

def _resize_with_pad_pil(image: Image.Image, height: int, width: int) -> np.ndarray:
    """
    PIL 이미지를 리사이즈하면서 패딩을 추가 (aspect ratio 유지)
    
    Args:
        image: PIL Image
        height: 목표 높이
        width: 목표 너비
    
    Returns:
        (height, width, 3) numpy array (RGB)
    """
    cur_width, cur_height = image.size
    if cur_width == width and cur_height == height:
        return np.asarray(image, dtype=np.uint8)
    
    # aspect ratio 유지하며 리사이즈
    ratio = max(cur_width / width, cur_height / height)
    resized_height = int(cur_height / ratio)
    resized_width = int(cur_width / ratio)
    resized_image = image.resize((resized_width, resized_height), Image.Resampling.LANCZOS)
    
    # 검은색 패딩 추가
    zero_image = Image.new(resized_image.mode, (width, height), 0)
    pad_height = max(0, int((height - resized_height) / 2))
    pad_width = max(0, int((width - resized_width) / 2))
    zero_image.paste(resized_image, (pad_width, pad_height))
    
    return np.asarray(zero_image, dtype=np.uint8)


def _images_to_fixed_chw_uint8(image_arrays: list, height: int, width: int) -> pa.Array:
    """
    이미지 배열을 OpenPI-safe uint8 고정크기 [C,H,W] 중첩 리스트로 변환
    (padding을 포함한 리사이즈)
    
    Args:
        image_arrays: list of (H, W, C) numpy arrays or PIL Images (RGB)
        height: 목표 높이 (padding 포함)
        width: 목표 너비 (padding 포함)
    
    Returns:
        PyArrow FixedSizeList array [3, H, W]
    """
    chw_list = []
    for img_arr in tqdm(image_arrays, desc='decode+resize images', leave=False):
        # numpy array이면 PIL로 변환, PIL Image면 바로 처리
        if isinstance(img_arr, Image.Image):
            img = img_arr.copy()
        elif isinstance(img_arr, np.ndarray):
            img = Image.fromarray(img_arr.astype(np.uint8))
        else:
            raise TypeError(f'Unexpected image type: {type(img_arr)}')
        
        # RGB로 변환
        if img.mode != 'RGB':
            img = img.convert('RGB')
        
        # 패딩을 포함한 리사이즈: aspect ratio 유지
        arr = _resize_with_pad_pil(img, height, width)
        if arr.ndim != 3 or arr.shape[2] != 3:
            raise ValueError(f'Expected HWC RGB image, got shape {arr.shape}')
        
        # CHW로 변환
        chw = np.transpose(arr, (2, 0, 1))
        chw_list.append(chw)
    
    # 배열 스택
    if not chw_list:
        # 빈 배열 반환
        values = pa.array([], type=pa.uint8())
        lvl_w = pa.FixedSizeListArray.from_arrays(values, width)
        lvl_h = pa.FixedSizeListArray.from_arrays(lvl_w, height)
        lvl_c = pa.FixedSizeListArray.from_arrays(lvl_h, 3)
        return lvl_c
    
    stacked = np.stack(chw_list, axis=0)  # N,C,H,W
    if stacked.shape[1] != 3 or stacked.shape[2] != height or stacked.shape[3] != width:
        raise ValueError(f'Unexpected stacked shape: {stacked.shape}')
    
    # PyArrow 중첩 FixedSizeList 생성
    flat = stacked.reshape(-1).tolist()
    values = pa.array(flat, type=pa.uint8())
    lvl_w = pa.FixedSizeListArray.from_arrays(values, width)
    lvl_h = pa.FixedSizeListArray.from_arrays(lvl_w, height)
    lvl_c = pa.FixedSizeListArray.from_arrays(lvl_h, 3)
    return lvl_c

def _zeros_fixed_chw_uint8(n: int, height: int, width: int) -> pa.Array:
    """
    0으로 채워진 OpenPI-safe uint8 [C,H,W] 배열 생성 (길이 n)
    """
    if n <= 0:
        values = pa.array([], type=pa.uint8())
        lvl_w = pa.FixedSizeListArray.from_arrays(values, width)
        lvl_h = pa.FixedSizeListArray.from_arrays(lvl_w, height)
        lvl_c = pa.FixedSizeListArray.from_arrays(lvl_h, 3)
        return lvl_c
    
    stacked = np.zeros((n, 3, height, width), dtype=np.uint8)
    flat = stacked.reshape(-1).tolist()
    values = pa.array(flat, type=pa.uint8())
    lvl_w = pa.FixedSizeListArray.from_arrays(values, width)
    lvl_h = pa.FixedSizeListArray.from_arrays(lvl_w, height)
    lvl_c = pa.FixedSizeListArray.from_arrays(lvl_h, 3)
    return lvl_c

print('Helper functions loaded.')


Helper functions loaded.
RAW_H5_IS_BGR = False (BGR→RGB 변환 비활성화 (raw 데이터가 이미 RGB))


## 4. 메인 변환 함수

In [12]:
def convert_h5_to_v21_dataset():
    """
    Raw H5 파일을 직접 LeRobotDataset v2.1 형식으로 변환
    """
    logging.info(f"'{RAW_H5_DIR}'에서 .h5 파일 변환 시작...")
    
    # 출력 디렉토리 정리
    if OUTPUT_V21_DATASET_DIR.exists():
        logging.info(f"기존 데이터셋 디렉토리 '{OUTPUT_V21_DATASET_DIR}' 삭제 중...")
        shutil.rmtree(OUTPUT_V21_DATASET_DIR)
    
    # 디렉토리 생성
    (OUTPUT_V21_DATASET_DIR / 'meta').mkdir(parents=True, exist_ok=True)
    (OUTPUT_V21_DATASET_DIR / 'data').mkdir(parents=True, exist_ok=True)
    
    # v2.1 features 정의
    v21_features = {
        'state': {'dtype': 'float32', 'shape': (16,), 'names': None},
        'actions': {'dtype': 'float32', 'shape': (16,), 'names': None},
        'prompt': {'dtype': 'string', 'shape': [1], 'names': None},
        
        'head_image': {
            'dtype': 'uint8',
            'shape': [3, IMAGE_HEIGHT, IMAGE_WIDTH],
            'names': ['channels', 'height', 'width'],
        },
        'left_wrist_image': {
            'dtype': 'uint8',
            'shape': [3, IMAGE_HEIGHT, IMAGE_WIDTH],
            'names': ['channels', 'height', 'width'],
        },
        'right_wrist_image': {
            'dtype': 'uint8',
            'shape': [3, IMAGE_HEIGHT, IMAGE_WIDTH],
            'names': ['channels', 'height', 'width'],
        },
        
        # LeRobot 표준 필드
        'timestamp': {'dtype': 'float32', 'shape': [1], 'names': None},
        'frame_index': {'dtype': 'int64', 'shape': [1], 'names': None},
        'episode_index': {'dtype': 'int64', 'shape': [1], 'names': None},
        'index': {'dtype': 'int64', 'shape': [1], 'names': None},
        # lerobot v2.1 필수: 각 프레임의 task 인덱스
        'task_index': {'dtype': 'int64', 'shape': [1], 'names': None},
        
        # 추가 필드
        'observation.state': {'dtype': 'float32', 'shape': (16,), 'names': None},
        'action': {'dtype': 'float32', 'shape': (16,), 'names': None},
        'is_first': {'dtype': 'bool', 'shape': [1], 'names': None},
        'is_last': {'dtype': 'bool', 'shape': [1], 'names': None},
        'is_terminal': {'dtype': 'bool', 'shape': [1], 'names': None},
    }
    
    # info.json 생성
    v21_info = {
        'codebase_version': 'v2.1',
        'robot_type': 'rby1',
        'total_episodes': 0,  # 아래에서 업데이트
        'total_frames': 0,
        'total_tasks': 1,
        'total_chunks': 0,
        'chunks_size': CHUNKS_SIZE,
        'fps': DATASET_FPS,
        'splits': {'train': None},  # 아래에서 업데이트
        'data_path': 'data/chunk-{episode_chunk:03d}/episode_{episode_index:06d}.parquet',
        'video_path': None,
        'features': v21_features,
    }
    
    # H5 파일 찾기
    h5_files = sorted(list(RAW_H5_DIR.glob("episode_*/*.h5")))
    if not h5_files:
        logging.error(f"'{RAW_H5_DIR}' 하위 episode_*/ 디렉토리에서 .h5 파일을 찾을 수 없습니다.")
        return
    
    logging.info(f"총 {len(h5_files)}개의 H5 파일을 찾았습니다.")
    
    # 첫 번째 H5 파일에서 이미지 키 감지
    image_keys_map = {'head': None, 'left': None, 'right': None}
    with h5py.File(h5_files[0], 'r') as f:
        # 사용 가능한 이미지 키 찾기
        for key in f.keys():
            key_lower = str(key).lower()
            if 'head' in key_lower:
                image_keys_map['head'] = key
            elif 'left' in key_lower:
                image_keys_map['left'] = key
            elif 'right' in key_lower:
                image_keys_map['right'] = key
    
    logging.info(f"감지된 이미지 키: head={image_keys_map['head']}, left={image_keys_map['left']}, right={image_keys_map['right']}")
    
    # 에피소드별 처리
    episodes_jsonl_rows = []
    episodes_stats_jsonl_rows = []
    max_chunk_index = -1
    total_frames = 0
    
    for episode_idx, h5_path in enumerate(h5_files):
        logging.info(f"에피소드 {episode_idx+1}/{len(h5_files)}: '{h5_path.name}' 변환 시작...")
        try:
            with h5py.File(h5_path, 'r') as f:
                # frame 수 (head 이미지로부터)
                if image_keys_map['head']:
                    num_frames = f[image_keys_map['head']]['image'].shape[0]
                else:
                    raise ValueError("head 이미지를 찾을 수 없습니다.")
                
                # 프레임 단위 데이터 수집
                frame_data_dict = {}
                head_image_data_list = []
                left_image_data_list = []
                right_image_data_list = []
                
                for frame_idx in range(num_frames):
                    frame_data = {}
                    
                    # 메타데이터
                    frame_data['is_first'] = np.array([frame_idx == 0], dtype=bool)
                    frame_data['is_last'] = np.array([frame_idx == num_frames - 1], dtype=bool)
                    frame_data['is_terminal'] = np.array([frame_idx == num_frames - 1], dtype=bool)
                    frame_data['prompt'] = conversion_prompt  # config.yaml에서 로드한 prompt
                    frame_data['frame_index'] = np.array([frame_idx], dtype=np.int64)
                    frame_data['episode_index'] = np.array([episode_idx], dtype=np.int64)
                    frame_data['index'] = np.array([total_frames + frame_idx], dtype=np.int64)
                    frame_data['timestamp'] = np.array([frame_idx / DATASET_FPS], dtype=np.float32)
                    
                    # 이미지 데이터 수집
                    if image_keys_map['head']:
                        head_img = f[image_keys_map['head']]['image'][frame_idx]
                        head_image_data_list.append(head_img)
                    
                    if image_keys_map['left']:
                        left_img = f[image_keys_map['left']]['image'][frame_idx]
                        left_image_data_list.append(left_img)
                    
                    if image_keys_map['right']:
                        right_img = f[image_keys_map['right']]['image'][frame_idx]
                        right_image_data_list.append(right_img)
                    
                    # 상태 데이터
                    base_state = f['samples/base_state'][frame_idx].astype(np.float32)
                    gripper_state = f['samples/gripper_state'][frame_idx].astype(np.float32)
                    robot_position = f['samples/robot_position'][frame_idx].astype(np.float32)
                    state = np.concatenate([robot_position[8:22], gripper_state]).astype(np.float32)
                    frame_data['observation.state'] = state
                    frame_data['state'] = state
                    
                    # 액션 데이터
                    gripper_target = f['samples/gripper_target'][frame_idx].astype(np.float32)
                    robot_target_joints = f['samples/robot_target_joints'][frame_idx].astype(np.float32)
                    action = np.concatenate([robot_target_joints[8:22], gripper_target]).astype(np.float32)
                    frame_data['action'] = action
                    frame_data['actions'] = action
                    
                    # 프레임 데이터 저장
                    if frame_idx not in frame_data_dict:
                        frame_data_dict[frame_idx] = {}
                    frame_data_dict[frame_idx].update(frame_data)
                
                # 이미지를 PyArrow 배열로 변환
                head_image_col = _images_to_fixed_chw_uint8(head_image_data_list, IMAGE_HEIGHT, IMAGE_WIDTH) if head_image_data_list else _zeros_fixed_chw_uint8(num_frames, IMAGE_HEIGHT, IMAGE_WIDTH)
                
                left_image_col = _images_to_fixed_chw_uint8(left_image_data_list, IMAGE_HEIGHT, IMAGE_WIDTH) if left_image_data_list else _zeros_fixed_chw_uint8(num_frames, IMAGE_HEIGHT, IMAGE_WIDTH)
                
                right_image_col = _images_to_fixed_chw_uint8(right_image_data_list, IMAGE_HEIGHT, IMAGE_WIDTH) if right_image_data_list else _zeros_fixed_chw_uint8(num_frames, IMAGE_HEIGHT, IMAGE_WIDTH)
                
                # 다른 컬럼들을 PyArrow 배열로 변환
                columns = {}
                for key in ['is_first', 'is_last', 'is_terminal', 'timestamp', 'frame_index', 'episode_index', 'index']:
                    col_data = [frame_data_dict[i][key] for i in range(num_frames)]
                    columns[key] = pa.array(col_data)
                
                for key in ['observation.state', 'action', 'state', 'actions']:
                    col_data = [frame_data_dict[i][key] for i in range(num_frames)]
                    columns[key] = pa.array(col_data)
                
                # prompt 컬럼 (config.yaml의 conversion_prompt 사용)
                columns['prompt'] = pa.array([conversion_prompt] * num_frames, type=pa.string())
                # task_index 컬럼 (lerobot v2.1 필수: item["task_index"] 참조)
                # tasks.jsonl의 task_index와 일치해야 함 (단일 task이므로 모두 0)
                columns['task_index'] = pa.array([0] * num_frames, type=pa.int64())
                
                # 이미지 컬럼 추가
                columns['head_image'] = head_image_col
                columns['left_wrist_image'] = left_image_col
                columns['right_wrist_image'] = right_image_col
                
                # PyArrow Table 생성
                out_tbl = pa.table(columns)
                out_tbl = out_tbl.replace_schema_metadata(None)
                
                # 청크 및 저장
                chunk_index = episode_idx // CHUNKS_SIZE
                max_chunk_index = max(max_chunk_index, int(chunk_index))
                out_chunk_dir = OUTPUT_V21_DATASET_DIR / 'data' / f'chunk-{chunk_index:03d}'
                out_chunk_dir.mkdir(parents=True, exist_ok=True)
                out_path = out_chunk_dir / f'episode_{episode_idx:06d}.parquet'
                pq.write_table(out_tbl, out_path, compression='zstd')
                
                logging.info(f"'{h5_path.name}' 저장 완료: {out_path}")
                
                # 에피소드 정보 기록
                episodes_jsonl_rows.append({
                    'episode_index': int(episode_idx),
                    'length': int(num_frames),
                    'tasks': [conversion_prompt],
                })
                
                # 에피소드 통계 (기본값)
                episodes_stats_jsonl_rows.append({
                    'episode_index': int(episode_idx),
                    'stats': {},
                })
                
                total_frames += num_frames
        
        except Exception as e:
            logging.error(f"'{h5_path.name}' 처리 중 오류 발생: {e}", exc_info=True)
            continue
    
    # 메타데이터 파일 작성
    v21_info['total_episodes'] = len(h5_files)
    v21_info['total_frames'] = total_frames
    v21_info['total_chunks'] = max_chunk_index + 1 if max_chunk_index >= 0 else 0
    v21_info['splits']['train'] = f'0:{len(h5_files)}'
    _write_json(OUTPUT_V21_DATASET_DIR / 'meta/info.json', v21_info)
    
    _write_jsonl(OUTPUT_V21_DATASET_DIR / 'meta/episodes.jsonl', episodes_jsonl_rows)
    _write_jsonl(OUTPUT_V21_DATASET_DIR / 'meta/episodes_stats.jsonl', episodes_stats_jsonl_rows)
    _write_jsonl(OUTPUT_V21_DATASET_DIR / 'meta/tasks.jsonl', [
        {'task_index': 0, 'task': conversion_prompt}
    ])
    
    logging.info(f"LeRobotDataset v2.1 변환 완료. '{OUTPUT_V21_DATASET_DIR}'에 저장되었습니다.")
    logging.info(f"총 에피소드: {len(h5_files)}, 총 프레임: {total_frames}")

# 실행
if __name__ == "__main__":
    convert_h5_to_v21_dataset()


2026-03-05 16:13:41,642 - INFO - '/media/hyunjin/T7/rby1_demo/PuttingCupintotheDishV2'에서 .h5 파일 변환 시작...
2026-03-05 16:13:41,643 - INFO - 기존 데이터셋 디렉토리 '/media/hyunjin/T7/rby1_demo/LeRobotDataset_v2/PuttingCupintotheDishV2' 삭제 중...
2026-03-05 16:13:41,733 - INFO - 총 100개의 H5 파일을 찾았습니다.
2026-03-05 16:13:41,734 - INFO - 감지된 이미지 키: head=head_rgb, left=left_rgb, right=right_rgb
2026-03-05 16:13:41,734 - INFO - 에피소드 1/100: 'demo_0.h5' 변환 시작...


decode+resize images:   0%|          | 0/211 [00:00<?, ?it/s]

decode+resize images:   0%|          | 0/211 [00:00<?, ?it/s]

decode+resize images:   0%|          | 0/211 [00:00<?, ?it/s]

2026-03-05 16:13:47,583 - INFO - 'demo_0.h5' 저장 완료: /media/hyunjin/T7/rby1_demo/LeRobotDataset_v2/PuttingCupintotheDishV2/data/chunk-000/episode_000000.parquet
2026-03-05 16:13:47,583 - INFO - 에피소드 2/100: 'demo_1.h5' 변환 시작...


...

2026-03-05 16:22:24,580 - INFO - 'demo_98.h5' 저장 완료: /media/hyunjin/T7/rby1_demo/LeRobotDataset_v2/PuttingCupintotheDishV2/data/chunk-000/episode_000098.parquet
2026-03-05 16:22:24,580 - INFO - 에피소드 100/100: 'demo_99.h5' 변환 시작...


decode+resize images:   0%|          | 0/136 [00:00<?, ?it/s]

decode+resize images:   0%|          | 0/136 [00:00<?, ?it/s]

decode+resize images:   0%|          | 0/136 [00:00<?, ?it/s]

2026-03-05 16:22:28,576 - INFO - 'demo_99.h5' 저장 완료: /media/hyunjin/T7/rby1_demo/LeRobotDataset_v2/PuttingCupintotheDishV2/data/chunk-000/episode_000099.parquet
2026-03-05 16:22:28,577 - INFO - LeRobotDataset v2.1 변환 완료. '/media/hyunjin/T7/rby1_demo/LeRobotDataset_v2/PuttingCupintotheDishV2'에 저장되었습니다.
2026-03-05 16:22:28,577 - INFO - 총 에피소드: 100, 총 프레임: 18544


## 6. 검증

In [13]:
# 📁 최종 폴더 구조 확인
import os
import glob

def print_tree(directory, prefix="", max_depth=3, current_depth=0):
    """재귀적으로 디렉토리 구조 출력"""
    if current_depth >= max_depth:
        return
    
    try:
        entries = sorted(os.listdir(directory))
    except PermissionError:
        return
    
    dirs = [e for e in entries if os.path.isdir(os.path.join(directory, e))]
    files = [e for e in entries if os.path.isfile(os.path.join(directory, e))]
    
    # 파일 출력
    for i, file in enumerate(files):
        is_last = (i == len(files) - 1) and len(dirs) == 0
        symbol = "└── " if is_last else "├── "
        print(f"{prefix}{symbol}{file}")
    
    # 디렉토리 출력
    for i, dir_name in enumerate(dirs):
        is_last = i == len(dirs) - 1
        symbol = "└── " if is_last else "├── "
        print(f"{prefix}{symbol}{dir_name}/")
        
        extension = "    " if is_last else "│   "
        print_tree(os.path.join(directory, dir_name), prefix + extension, max_depth, current_depth + 1)

print("\n" + "="*60)
print("📁 실제 생성된 데이터셋 폴더 구조")
print("="*60)
print(f"\n{OUTPUT_V21_DATASET_DIR}/")
print_tree(str(OUTPUT_V21_DATASET_DIR), max_depth=3)

print("\n" + "="*60)
print("📊 구조 요약")
print("="*60)

meta_dir = OUTPUT_V21_DATASET_DIR / 'meta'
data_dir = OUTPUT_V21_DATASET_DIR / 'data'
video_dir = OUTPUT_V21_DATASET_DIR / 'videos'

# Meta 파일들
meta_files = list(meta_dir.glob('*')) if meta_dir.exists() else []
print(f"\n✅ Meta 디렉토리: {meta_dir.exists()}")
for f in sorted(meta_files):
    print(f"   └─ {f.name}")

# Data 파일들
data_chunks = list(data_dir.glob('chunk-*')) if data_dir.exists() else []
print(f"\n✅ Data 디렉토리: {data_dir.exists()}")
print(f"   총 청크: {len(data_chunks)}")
for chunk in sorted(data_chunks)[:3]:  # 처음 3개만
    parquets = list(chunk.glob('*.parquet'))
    print(f"   └─ {chunk.name}/ ({len(parquets)} parquet files)")

# Videos 폴더
print(f"\n❓ Videos 디렉토리: {video_dir.exists()}")
if not video_dir.exists():
    print(f"   → WRITE_VIDEOS = False이므로 생성되지 않음")
    print(f"   → 이미지는 parquet의 head_image/left_wrist_image/right_wrist_image 컬럼에 포함됨")

print("\n" + "="*60)
print("✅ 최종 데이터셋 형식")
print("="*60)
print(f"""
{task_name}/
├── data/
│   └── chunk-000/
│       ├── episode_000000.parquet
│       ├── episode_000001.parquet
│       └── ... (이미지 데이터 포함: head_image, left_wrist_image, right_wrist_image)
│
├── meta/
│   ├── info.json
│   ├── episodes.jsonl
│   ├── episodes_stats.jsonl
│   └── tasks.jsonl
│
└── (videos/ 폴더는 없음 - 이미지는 parquet에 저장)

📌 주요 특징:
  • 이미지: parquet 컬럼에 OpenPI-safe uint8 nested list [C,H,W] 형식으로 저장
  • 크기: 224x224 (padding 포함, aspect ratio 유지)
  • 컬럼: head_image, left_wrist_image, right_wrist_image
  • 형식: PyArrow FixedSizeList (3차원 배열)
""")



📁 실제 생성된 데이터셋 폴더 구조

/media/hyunjin/T7/rby1_demo/LeRobotDataset_v2/PuttingCupintotheDishV2/
├── data/
│   └── chunk-000/
│       ├── episode_000000.parquet
│       ├── episode_000001.parquet
│       ├── episode_000002.parquet
│       ├── episode_000003.parquet
│       ├── episode_000004.parquet
│       ├── episode_000005.parquet
│       ├── episode_000006.parquet
│       ├── episode_000007.parquet
│       ├── episode_000008.parquet
│       ├── episode_000009.parquet
│       ├── episode_000010.parquet
│       ├── episode_000011.parquet
│       ├── episode_000012.parquet
│       ├── episode_000013.parquet
│       ├── episode_000014.parquet
│       ├── episode_000015.parquet
│       ├── episode_000016.parquet
│       ├── episode_000017.parquet
│       ├── episode_000018.parquet
│       ├── episode_000019.parquet
│       ├── episode_000020.parquet
│       ├── episode_000021.parquet
│       ├── episode_000022.parquet
│       ├── episode_000023.parquet
│       ├── episode_000024.parquet
│  

## 7. Hugging Face Hub 업로드

In [14]:
# 사전에 huggingface-cli login 필요
import shutil as _shutil
from pathlib import Path as _Path
from huggingface_hub import HfApi, create_repo, upload_folder

REPO_ID = f"{user_name}/{task_name}"
PRIVATE = False

api = HfApi()

# 1. 레포지토리 생성
create_repo(repo_id=REPO_ID, repo_type='dataset', private=PRIVATE, exist_ok=True)
print(f'\u2705 Repo ready: https://huggingface.co/datasets/{REPO_ID}')

# 2. 업로드
print(f'\n\U0001f680 업로드 중...')
print(f'   로컬: {OUTPUT_V21_DATASET_DIR}')
print(f'   허브:  {REPO_ID}')
res = upload_folder(
    repo_id=REPO_ID,
    repo_type='dataset',
    folder_path=str(OUTPUT_V21_DATASET_DIR),
    path_in_repo='',
    commit_message='Update: LeRobotDataset v2.1 (RGB)',
)
print(f'\u2705 업로드 완료: {res}')

# 3. v2.1 태그 갱신 (기존 태그 삭제 후 재생성)
try:
    api.delete_tag(REPO_ID, tag='v2.1', repo_type='dataset')
    print('\U0001f5d1\ufe0f  기존 v2.1 태그 삭제')
except Exception:
    pass

api.create_tag(REPO_ID, tag='v2.1', repo_type='dataset')
print('\u2705 태그 v2.1 생성 완료')

# 4. 로컬 HuggingFace 캐시 삭제 (구버전 캐시 제거)
hf_cache_path = _Path.home() / '.cache' / 'huggingface' / 'lerobot' / user_name / task_name
if hf_cache_path.exists():
    _shutil.rmtree(hf_cache_path)
    print(f'\U0001f5d1\ufe0f  로컬 HF 캐시 삭제: {hf_cache_path}')
else:
    print(f'\u2139\ufe0f  로컬 HF 캐시 없음 (정상): {hf_cache_path}')

print(f'\n\U0001f389 완료! 학습 재시작 가능')
print(f'   cd /home/hyunjin/rby1_ws/openpi')
print(f'   uv run python scripts/compute_norm_stats.py --config-name pi05_rby1')
print(f'   uv run python scripts/train.py pi05_rby1 --exp-name {task_name}')


✅ Repo ready: meat000124/PuttingCupintotheDishV2

🚀 업로드 중... (/media/hyunjin/T7/rby1_demo/LeRobotDataset_v2/PuttingCupintotheDishV2 → meat000124/PuttingCupintotheDishV2)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ 업로드 완료: https://huggingface.co/datasets/meat000124/PuttingCupintotheDishV2/tree/main/
🗑️  기존 v2.1 태그 삭제
✅ 태그 v2.1 생성 완료
🗑️  로컬 HF 캐시 삭제: /home/hyunjin/.cache/huggingface/lerobot/meat000124/PuttingCupintotheDishV2

🎉 완료! 학습 재시작 가능
   XLA_PYTHON_CLIENT_MEM_FRACTION=0.9 uv run scripts/train.py pi05_rby1 --exp-name PuttingCupintotheDishV2
